# 北上广深租房市场数据分析

## 第六部分：行政区整租市场分析

In [2]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/rent_cleaned.csv")

df = pd.read_csv(DATA_PATH)

print("数据规模：", df.shape)

数据规模： (11978, 26)


## 一、建立整租分析数据

In [3]:
# 单独建立整租分析数据
entire_df = df[
    df["type"] == "整租"
].copy()

district_count = len(
    entire_df[["city", "dist"]].drop_duplicates()
)

print("整租房源数量：", len(entire_df))
print("整租城市数量：", entire_df["city"].nunique())
print("整租行政区数量：", district_count)

整租房源数量： 11564
整租城市数量： 4
整租行政区数量： 49


## 二、检查行政区样本分布

In [4]:
# 统计每个城市各行政区的整租房源数量
district_counts = entire_df.groupby(
    ["city", "dist"]
).size()

district_counts = district_counts.reset_index()

district_counts.columns = [
    "city",
    "dist",
    "整租房源数量"
]

district_counts = district_counts.sort_values(
    by=["city", "整租房源数量"],
    ascending=[True, False]
)

district_counts = district_counts.reset_index(
    drop=True
)

district_counts

,city,dist,整租房源数量
0,上海,浦东,791
1,上海,闵行,286
2,上海,松江,261
3,上海,徐汇,214
4,上海,普陀,191
5,上海,嘉定,189
6,上海,长宁,180
7,上海,青浦,173
8,上海,宝山,167
9,上海,黄浦,111


In [5]:
# 汇总各城市行政区的整租样本量
district_sample_summary = district_counts.groupby(
    "city"
)["整租房源数量"].describe().round(2)

district_sample_summary

,count,mean,std,min,25%,50%,75%,max
city,,,,,,,,
上海,17.0,176.00,178.00,1.0,90.0,167.0,191.0,791.0
北京,13.0,230.15,262.18,56.0,91.0,170.0,221.0,1047.0
广州,9.0,322.00,219.98,21.0,220.0,263.0,506.0,662.0
深圳,10.0,268.20,235.05,2.0,36.5,272.5,490.0,575.0


In [7]:
# 找出整租样本少于30条的行政区
small_sample_districts = district_counts[
    district_counts["整租房源数量"] < 30
].copy()

small_sample_districts

,city,dist,整租房源数量
15,上海,金山,4
16,上海,崇明,1
38,广州,从化,21
46,深圳,盐田区,26
47,深圳,坪山区,9
48,深圳,大鹏新区,2


In [9]:
# 找出整租样本不少于30条的行政区
eligible_districts = district_counts[
    district_counts["整租房源数量"] >= 30
].copy()

# 重新生成连续行号
eligible_districts = eligible_districts.reset_index(
    drop=True
)

print("全部行政区数量：", len(district_counts))
print("参与排名的行政区数量：", len(eligible_districts))
print("样本不足的行政区数量：", len(small_sample_districts))

eligible_districts

全部行政区数量： 49
参与排名的行政区数量： 43
样本不足的行政区数量： 6


,city,dist,整租房源数量
0,上海,浦东,791
1,上海,闵行,286
2,上海,松江,261
3,上海,徐汇,214
4,上海,普陀,191
5,上海,嘉定,189
6,上海,长宁,180
7,上海,青浦,173
8,上海,宝山,167
9,上海,黄浦,111


## 三、行政区整租月租金比较

In [14]:
# 计算各行政区的整租房源数量
district_rent_count = entire_df.groupby(
    ["city", "dist"]
)["rent_price_num"].count()

In [15]:
# 计算各行政区的整租平均月租金
district_rent_mean = entire_df.groupby(
    ["city", "dist"]
)["rent_price_num"].mean().round(2)

In [16]:
# 计算各行政区的整租月租金中位数
district_rent_median = entire_df.groupby(
    ["city", "dist"]
)["rent_price_num"].median()


In [17]:
# 把三项结果组合成一张表
district_rent = pd.DataFrame({
    "整租房源数量": district_rent_count,
    "整租平均月租金": district_rent_mean,
    "整租月租金中位数": district_rent_median
})

district_rent = district_rent.reset_index()

district_rent = district_rent.sort_values(  # 排序优先级是：先按照city排序，让同一城市的行政区集中在一起；如果城市相同，再按照“整租月租金中位数”排序。
    by=["city", "整租月租金中位数"],
    ascending=[True, False]
)

district_rent = district_rent.reset_index(
    drop=True
)

district_rent

,city,dist,整租房源数量,整租平均月租金,整租月租金中位数
0,上海,静安,91,14999.45,13000.0
1,上海,黄浦,111,17476.05,12500.0
2,上海,长宁,180,13827.78,12000.0
3,上海,徐汇,214,10338.60,7000.0
4,上海,虹口,78,9169.08,6900.0
5,上海,普陀,191,8175.55,6500.0
6,上海,闸北,90,11217.78,6400.0
7,上海,闵行,286,9362.59,6250.0
8,上海,浦东,791,10122.47,6200.0
9,上海,杨浦,108,7263.89,5550.0


In [18]:
# 筛选可以参与月租金排名的行政区
district_rent_eligible = district_rent[
    district_rent["整租房源数量"] >= 30
].copy()

# 重新生成连续行号
district_rent_eligible = district_rent_eligible.reset_index(
    drop=True
)

print(
    "参与月租金排名的行政区数量：",
    len(district_rent_eligible)
)

district_rent_eligible

参与月租金排名的行政区数量： 43


,city,dist,整租房源数量,整租平均月租金,整租月租金中位数
0,上海,静安,91,14999.45,13000.0
1,上海,黄浦,111,17476.05,12500.0
2,上海,长宁,180,13827.78,12000.0
3,上海,徐汇,214,10338.60,7000.0
4,上海,虹口,78,9169.08,6900.0
5,上海,普陀,191,8175.55,6500.0
6,上海,闸北,90,11217.78,6400.0
7,上海,闵行,286,9362.59,6250.0
8,上海,浦东,791,10122.47,6200.0
9,上海,杨浦,108,7263.89,5550.0


In [19]:
# 找出每个城市月租金中位数最高的3个行政区
top_district_rent = district_rent_eligible.groupby(
    "city"
).head(3).copy()

top_district_rent = top_district_rent.reset_index(
    drop=True
)

print("结果行数：", len(top_district_rent))

top_district_rent

结果行数： 12


,city,dist,整租房源数量,整租平均月租金,整租月租金中位数
0,上海,静安,91,14999.45,13000.0
1,上海,黄浦,111,17476.05,12500.0
2,上海,长宁,180,13827.78,12000.0
3,北京,海淀,328,11191.13,8200.0
4,北京,朝阳,1047,11845.77,8000.0
5,北京,东城,64,8948.44,7800.0
6,广州,越秀,254,5859.53,4500.0
7,广州,天河,506,5321.14,4300.0
8,广州,海珠,263,4571.92,3800.0
9,深圳,南山区,530,12940.88,8500.0


## 四、行政区整租每平方米租金比较

In [20]:
# 计算各行政区的整租房源数量
district_per_sqm_count = entire_df.groupby(
    ["city", "dist"]
)["rent_price_per_sqm"].count()

In [21]:
# 计算各行政区的整租平均每平方米租金
district_per_sqm_mean = entire_df.groupby(
    ["city", "dist"]
)["rent_price_per_sqm"].mean().round(2)

In [22]:
# 计算各行政区的整租每平方米租金中位数
district_per_sqm_median = entire_df.groupby(
    ["city", "dist"]
)["rent_price_per_sqm"].median().round(2)


In [23]:
# 把三项结果组合成一张表
district_per_sqm = pd.DataFrame({
    "整租房源数量": district_per_sqm_count,
    "整租平均每平方米租金": district_per_sqm_mean,
    "整租每平方米租金中位数": district_per_sqm_median
})

district_per_sqm = district_per_sqm.reset_index()  # 第一次不写drop=True：把city、dist从分组索引变成普通字段并保留。

district_per_sqm = district_per_sqm.sort_values(
    by=["city", "整租每平方米租金中位数"],
    ascending=[True, False]
)

district_per_sqm = district_per_sqm.reset_index(
    drop=True  # 第二次写drop=True：丢掉排序后的旧行号。
)

district_per_sqm

,city,dist,整租房源数量,整租平均每平方米租金,整租每平方米租金中位数
0,上海,静安,91,155.63,153.06
1,上海,黄浦,111,145.40,133.33
2,上海,长宁,180,133.87,128.80
3,上海,徐汇,214,125.90,115.58
4,上海,闸北,90,117.01,102.19
5,上海,虹口,78,106.47,100.00
6,上海,普陀,191,97.02,96.43
7,上海,杨浦,108,98.63,94.36
8,上海,浦东,791,85.96,81.36
9,上海,闵行,286,75.14,69.36


In [24]:
# 筛选可以参与单位面积租金排名的行政区
district_per_sqm_eligible = district_per_sqm[
    district_per_sqm["整租房源数量"] >= 30
].copy()

district_per_sqm_eligible = (
    district_per_sqm_eligible.reset_index(
        drop=True
    )
)

print(
    "参与单位面积租金排名的行政区数量：",
    len(district_per_sqm_eligible)
)

district_per_sqm_eligible

参与单位面积租金排名的行政区数量： 43


,city,dist,整租房源数量,整租平均每平方米租金,整租每平方米租金中位数
0,上海,静安,91,155.63,153.06
1,上海,黄浦,111,145.40,133.33
2,上海,长宁,180,133.87,128.80
3,上海,徐汇,214,125.90,115.58
4,上海,闸北,90,117.01,102.19
5,上海,虹口,78,106.47,100.00
6,上海,普陀,191,97.02,96.43
7,上海,杨浦,108,98.63,94.36
8,上海,浦东,791,85.96,81.36
9,上海,闵行,286,75.14,69.36


In [25]:
# 找出每个城市单位面积租金中位数最高的3个行政区
top_district_per_sqm = district_per_sqm_eligible.groupby(
    "city"
).head(3).copy()

top_district_per_sqm = top_district_per_sqm.reset_index(
    drop=True
)

print("结果行数：", len(top_district_per_sqm))

top_district_per_sqm

结果行数： 12


,city,dist,整租房源数量,整租平均每平方米租金,整租每平方米租金中位数
0,上海,静安,91,155.63,153.06
1,上海,黄浦,111,145.40,133.33
2,上海,长宁,180,133.87,128.80
3,北京,东城,64,138.79,129.17
4,北京,西城,198,134.23,125.40
5,北京,海淀,328,121.23,116.22
6,广州,越秀,254,85.63,75.00
7,广州,天河,506,71.58,63.12
8,广州,海珠,263,63.83,60.00
9,深圳,南山区,530,132.63,121.67


## 五、统一检查行政区分析结果

In [26]:
# 统一检查第6天使用的数据和主要结果
print("清洗数据规模：", df.shape)
print("整租房源数量：", len(entire_df))
print("整租出租类型：", entire_df["type"].unique())

print("全部行政区数量：", len(district_counts))
print("参与排名的行政区数量：", len(district_per_sqm_eligible))
print("样本不足的行政区数量：", len(small_sample_districts))

print("月租金前3名结果行数：", len(top_district_rent))
print("单位面积租金前3名结果行数：", len(top_district_per_sqm))

print("\n各城市单位面积租金结果数量：")
print(
    top_district_per_sqm["city"].value_counts()
)

清洗数据规模： (11978, 26)
整租房源数量： 11564
整租出租类型： <StringArray>
['整租']
Length: 1, dtype: str
全部行政区数量： 49
参与排名的行政区数量： 43
样本不足的行政区数量： 6
月租金前3名结果行数： 12
单位面积租金前3名结果行数： 12

各城市单位面积租金结果数量：
city
上海    3
北京    3
广州    3
深圳    3
Name: count, dtype: int64


## 数据分析总结

### 数据基本情况

- 本次分析使用第3天生成的清洗数据，共11,978条住宅租房记录和26个字段。
- 行政区市场分析只使用11,564条整租房源，没有把整租和合租混合分析。
- 清洗数据共包含49个“城市＋行政区”组合。
- 为避免极少数房源代表整个行政区，正式排名只包括至少有30条整租样本的行政区。
- 共有43个行政区达到排名要求，6个行政区样本不足。
- 30条是本项目描述性排名使用的基础门槛，不是普遍适用的统计标准。
- 样本不足的行政区仍保留在完整统计结果中，没有删除房源或修改清洗数据。

### 主要分析结果

- 上海整租月租金中位数最高的3个合格行政区是静安、黄浦和长宁，分别为13,000元、12,500元和12,000元。
- 北京整租月租金中位数最高的3个合格行政区是海淀、朝阳和东城，分别为8,200元、8,000元和7,800元。
- 广州整租月租金中位数最高的3个合格行政区是越秀、天河和海珠，分别为4,500元、4,300元和3,800元。
- 深圳整租月租金中位数最高的3个合格行政区是南山区、福田区和罗湖区，分别为8,500元、7,400元和5,750元。
- 上海整租每平方米租金中位数最高的3个合格行政区是静安、黄浦和长宁，分别为153.06元、133.33元和128.80元。
- 北京整租每平方米租金中位数最高的3个合格行政区是东城、西城和海淀，分别为129.17元、125.40元和116.22元。
- 广州整租每平方米租金中位数最高的3个合格行政区是越秀、天河和海珠，分别为75.00元、63.12元和60.00元。
- 深圳整租每平方米租金中位数最高的3个合格行政区是南山区、福田区和罗湖区，分别为121.67元、116.76元和100.00元。
- 上海、广州和深圳的月租金前3名与每平方米租金前3名行政区相同。
- 北京的两项排名存在差异：朝阳进入月租金前3名，西城进入每平方米租金前3名，说明总租金排名会受到房屋面积影响。
- 以上结果描述当前清洗样本中的整租房源，不代表行政区全部租赁市场。